# 00 · Exploración de Landing con PySpark

Entrega 1: evidencia mínima de lectura con **esquema explícito** (sin `inferSchema`). Corre en local o en Google Colab.

En Colab: `!pip -q install pyspark==4.0.1`, montar Drive y apuntar `LANDING` a la carpeta descomprimida.

In [1]:
import os
from pyspark.sql import SparkSession, functions as F, types as T

LANDING = os.environ.get("CPA_LANDING", "../datalake/landing")
spark = (SparkSession.builder.appName("cpa-exploracion").master("local[*]")
         .config("spark.sql.shuffle.partitions", "8")
         .config("spark.sql.session.timeZone", "UTC")
         .getOrCreate())
spark.sparkContext.setLogLevel("ERROR")
spark.version

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


26/10/02 21:33:12 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


'4.0.1'

## 1. Maestros y facturación (batch) con esquema explícito

In [2]:
ORGS = T.StructType([T.StructField(c, t) for c, t in [
    ("org_id", T.StringType()), ("org_name", T.StringType()), ("industry", T.StringType()),
    ("hq_region", T.StringType()), ("plan_tier", T.StringType()), ("is_enterprise", T.BooleanType()),
    ("signup_date", T.DateType()), ("sales_rep", T.StringType()), ("lifecycle_stage", T.StringType()),
    ("marketing_source", T.StringType()), ("nps_score", T.DoubleType())]])
BILLING = T.StructType([T.StructField(c, t) for c, t in [
    ("invoice_id", T.StringType()), ("org_id", T.StringType()), ("month", T.DateType()),
    ("subtotal", T.DecimalType(12, 2)), ("credits", T.DecimalType(12, 2)), ("taxes", T.DecimalType(12, 2)),
    ("currency", T.StringType()), ("exchange_rate_to_usd", T.DoubleType())]])

orgs = spark.read.schema(ORGS).option("header", True).csv(f"{LANDING}/customers_orgs.csv")
billing = spark.read.schema(BILLING).option("header", True).csv(f"{LANDING}/billing_monthly.csv")
orgs.printSchema()
print("orgs:", orgs.count(), "| facturas:", billing.count())

root
 |-- org_id: string (nullable = true)
 |-- org_name: string (nullable = true)
 |-- industry: string (nullable = true)
 |-- hq_region: string (nullable = true)
 |-- plan_tier: string (nullable = true)
 |-- is_enterprise: boolean (nullable = true)
 |-- signup_date: date (nullable = true)
 |-- sales_rep: string (nullable = true)
 |-- lifecycle_stage: string (nullable = true)
 |-- marketing_source: string (nullable = true)
 |-- nps_score: double (nullable = true)



orgs: 80 | facturas: 240


In [3]:
# Hallazgo FX: facturas en USD con exchange_rate_to_usd != 1 (decisión abierta D-06)
(billing.groupBy("currency")
 .agg(F.count("*").alias("facturas"), F.min("exchange_rate_to_usd").alias("fx_min"),
      F.max("exchange_rate_to_usd").alias("fx_max"), F.sum(F.col("credits").isNull().cast("int")).alias("credits_nulos"),
      F.sum((F.col("subtotal") < 0).cast("int")).alias("subtotal_negativo"))
 .orderBy("currency").show())

+--------+--------+-------+-------+-------------+-----------------+
|currency|facturas| fx_min| fx_max|credits_nulos|subtotal_negativo|
+--------+--------+-------+-------+-------------+-----------------+
|     ARS|      51|0.00133|0.00162|           34|                2|
|     EUR|      29| 0.9981|1.19808|           14|                1|
|     USD|     160|0.85463|1.11791|           89|               10|
+--------+--------+-------+-------+-------------+-----------------+



## 2. Eventos de uso (JSONL) — esquema unificado v1/v2

In [4]:
EVENTS = T.StructType([T.StructField(c, t) for c, t in [
    ("event_id", T.StringType()), ("timestamp", T.TimestampType()), ("org_id", T.StringType()),
    ("resource_id", T.StringType()), ("service", T.StringType()), ("region", T.StringType()),
    ("metric", T.StringType()), ("value", T.StringType()), ("unit", T.StringType()),
    ("cost_usd_increment", T.DoubleType()), ("schema_version", T.IntegerType()),
    ("carbon_kg", T.DoubleType()), ("genai_tokens", T.LongType())]])

events = (spark.read.schema(EVENTS).json(f"{LANDING}/usage_events_stream")
          .withColumn("source_file", F.input_file_name())
          .withColumn("value_num", F.col("value").try_cast("double")))
print("eventos:", events.count(), "| particiones de lectura:", events.rdd.getNumPartitions())

eventos: 43200 | particiones de lectura: 10


In [5]:
# Evolución de esquema: v2 agrega carbon_kg y genai_tokens
(events.groupBy("schema_version")
 .agg(F.count("*").alias("eventos"), F.min("timestamp").alias("desde"), F.max("timestamp").alias("hasta"),
      F.count("carbon_kg").alias("con_carbon"), F.count("genai_tokens").alias("con_tokens"))
 .orderBy("schema_version").show(truncate=False))

+--------------+-------+-------------------+-------------------+----------+----------+
|schema_version|eventos|desde              |hasta              |con_carbon|con_tokens|
+--------------+-------+-------------------+-------------------+----------+----------+
|1             |10800  |2025-07-03 00:02:00|2025-07-17 23:56:00|0         |0         |
|2             |32400  |2025-07-18 00:01:00|2025-08-31 23:58:00|32400     |3132      |
+--------------+-------+-------------------+-------------------+----------+----------+



In [6]:
# Calidad: value string/nulo, unit nulo, costos negativos, duplicados
events.select(
    F.sum((F.col("value").isNotNull() & F.col("value_num").isNull()).cast("int")).alias("value_no_casteable"),
    F.sum(F.col("value").isNull().cast("int")).alias("value_nulo"),
    F.sum(F.col("unit").isNull().cast("int")).alias("unit_nulo"),
    F.sum((F.col("cost_usd_increment") < 0).cast("int")).alias("costo_negativo"),
    (F.count("*") - F.countDistinct("event_id")).alias("event_id_duplicados"),
).show()

+------------------+----------+---------+--------------+-------------------+
|value_no_casteable|value_nulo|unit_nulo|costo_negativo|event_id_duplicados|
+------------------+----------+---------+--------------+-------------------+
|                 0|       877|     2075|           216|                  0|
+------------------+----------+---------+--------------+-------------------+



In [7]:
# Llegada desordenada: rango temporal cubierto por cada archivo (hallazgo crítico para el watermark)
(events.groupBy("source_file")
 .agg(F.min("timestamp").alias("ts_min"), F.max("timestamp").alias("ts_max"))
 .withColumn("dias_cubiertos", F.datediff("ts_max", "ts_min"))
 .select(F.min("dias_cubiertos").alias("min_dias"), F.avg("dias_cubiertos").alias("avg_dias"))
 .show())

+--------+--------+
|min_dias|avg_dias|
+--------+--------+
|      59|    59.0|
+--------+--------+



In [8]:
# Costo por servicio (base para el z robusto por servicio+métrica, D-08)
(events.groupBy("service")
 .agg(F.count("*").alias("eventos"), F.round(F.sum("cost_usd_increment"), 2).alias("costo_total"),
      F.expr("percentile_approx(cost_usd_increment, 0.5)").alias("p50"),
      F.expr("percentile_approx(cost_usd_increment, 0.99)").alias("p99"),
      F.max("cost_usd_increment").alias("max"))
 .orderBy(F.desc("costo_total")).show())

+----------+-------+-----------+------+-------+--------+
|   service|eventos|costo_total|   p50|    p99|     max|
+----------+-------+-----------+------+-------+--------+
|   compute|  12498|    61786.8|1.3733|13.4861|302.3459|
|     genai|   4158|   29544.43|2.0438|19.7267|317.4308|
|  database|   7419|   22635.46|0.9194| 8.3631|149.9091|
| analytics|   4590|   19708.76| 1.216|11.6259|249.3861|
|   storage|   7614|    9407.14|0.3536| 3.3358| 62.9922|
|networking|   6921|    4351.39|0.1904| 1.6749| 29.6319|
+----------+-------+-----------+------+-------+--------+



In [9]:
spark.stop()